# 05 - TF-IDF lexical baseline on MAGE

TF-IDF and logistic regression [52] on every training set: a performance floor for the other detectors, and an end-to-end test of the pipeline before any GPU time.

1. Checking the metrics
2. Load the corpus and training sets
3. One training set first
4. All training sets
5. Results
6. Save

In [1]:
import numpy as np
import pandas as pd

from logos.paths import DATA_DIR
from logos.project_data.mage import families
from logos.evaluation.folds import load_training_set, calibration_set, test_set
from logos.evaluation.metrics import calibrate_threshold, tpr_at_threshold, fpr_at_threshold, auroc
from logos.detectors.tf_idf import fit_lexical_baseline

MAGE_DIR = DATA_DIR / "mage"
LABELS = {0: "human", 1: "machine"}

## 1. Checking the metrics
Before any detector is run, the metrics are checked on fake scores with known answers: 10,000 "human" scores drawn around 0 and 10,000 "machine" scores around 2.

τ* is 2.311, close to the theoretical 2.326 above which 1% of human scores fall, and exactly 1.0% of the human scores are flagged. TPR is 0.379, matching the 0.378 expected for machine scores centred on 2, and AUROC is 0.922 against a theoretical 0.921. The example also previews H1: a detector with an AUROC of 0.922 catches only 37.9% of machine text at 1% FPR.

In [2]:
# 1. Fake scores: humans centred on 0, machines centred on 2
rng = np.random.default_rng(0)
humans = rng.normal(0, 1, 10000)
machines = rng.normal(2, 1, 10000)
scores = np.concatenate([humans, machines])
labels = np.concatenate([np.zeros(10000), np.ones(10000)])

# 2. Threshold from the human scores only, then each metric
tau = calibrate_threshold(humans, np.zeros(10000))
print("tau:", round(tau, 3))
print("fpr:", fpr_at_threshold(scores, labels, tau))
print("tpr:", round(tpr_at_threshold(scores, labels, tau), 3))
print("auroc:", round(auroc(scores, labels), 3))

tau: 2.311
fpr: 0.01
tpr: 0.379
auroc: 0.922


## 2. Load the corpus and training sets
The cleaned corpus and the training-set membership saved by `04` are loaded, so every detector uses identical data.

In [3]:
# 1. The cleaned corpus and which texts are in each training set
corpus = pd.read_parquet(MAGE_DIR / "corpus.parquet")
membership = pd.read_parquet(MAGE_DIR / "training_sets.parquet")
print("texts:", len(corpus), "| every text_id unique:", corpus["text_id"].is_unique)

# 2. Pairs per training set and seed (should match 04)
pairs = membership.groupby(["training_set", "seed"])["pair_id"].nunique().unstack()
print(pairs.to_string())

# 3. One training set joined back to its texts, split into train and tune parts
train_df = load_training_set(corpus, membership, seed=0, name="In-distribution")
print("texts in set:", len(train_df))
print(train_df.groupby(["part", train_df["label"].map(LABELS)]).size().unstack().to_string())

# 4. Calibration humans
calibration = calibration_set(corpus)
print("calibration humans:", len(calibration))

texts: 392385 | every text_id unique: True
seed                 0      1      2      3      4
training_set                                      
In-distribution  55148  55139  55159  55146  55162
LOFO BigScience  65416  65396  65437  65402  65405
LOFO EleutherAI  65617  65622  65615  65658  65656
LOFO FLAN-T5     62843  62854  62850  62885  62870
LOFO GLM-130B    66994  67017  67016  67026  66999
LOFO LLaMA       62487  62455  62483  62468  62478
LOFO OPT         55148  55139  55159  55146  55162
LOFO OpenAI GPT  63124  63145  63136  63103  63124
texts in set: 110296
label  human  machine
part                 
train  49633    49633
tune    5515     5515
calibration humans: 28407


The corpus loads with all 392,385 texts and a unique id for each, so joining training sets back to their texts cannot duplicate any. Pair counts match `04` for every set and seed. The seed 0 in-distribution set holds 110,296 texts, 49,633 pairs to train on and 5,515 pairs held back for tuning, each balanced between human and machine.

## 3. One training set first
The seed 0 in-distribution set is run on its own first, to check every step by eye before looping over all 40 runs.

In [4]:
# 1. Fit on the train part, choosing C on the tune part
train_part = train_df[train_df["part"] == "train"]
tune_part = train_df[train_df["part"] == "tune"]

model, best_c, accuracies = fit_lexical_baseline(
    train_part["text"], train_part["label"], tune_part["text"], tune_part["label"], seed=0)

print("features:", len(model.named_steps["tfidfvectorizer"].vocabulary_))
print("chosen C:", best_c)
for c, accuracy in accuracies.items():
    print(f"C = {c:g}: tune accuracy {accuracy:.4f}")

features: 443109
chosen C: 64
C = 0.015625: tune accuracy 0.6880
C = 0.03125: tune accuracy 0.7131
C = 0.0625: tune accuracy 0.7413
C = 0.125: tune accuracy 0.7716
C = 0.25: tune accuracy 0.7999
C = 0.5: tune accuracy 0.8311
C = 1: tune accuracy 0.8522
C = 2: tune accuracy 0.8692
C = 4: tune accuracy 0.8802
C = 8: tune accuracy 0.8887
C = 16: tune accuracy 0.8916
C = 32: tune accuracy 0.8927
C = 64: tune accuracy 0.8939


In [5]:
# 1. τ* from the calibration humans (score = probability of machine, so higher means machine)
calibration_scores = model.predict_proba(calibration["text"])[:, 1]
tau = calibrate_threshold(calibration_scores, calibration["label"])
print("tau:", round(tau, 4))

# 2. Score the whole test split once, then pick out each family's texts
test = test_set(corpus)
test = test.assign(score=model.predict_proba(test["text"])[:, 1])
humans = test[test["label"] == 0]
print("test FPR:", round(fpr_at_threshold(humans["score"], humans["label"], tau), 4))

rows = []
for family in families:
    family_test = test[(test["label"] == 0) | (test["family"] == family)]
    rows.append({"family": family,
                 "TPR@1%FPR": round(tpr_at_threshold(family_test["score"], family_test["label"], tau), 3),
                 "AUROC": round(auroc(family_test["score"], family_test["label"]), 3)})
print(pd.DataFrame(rows).to_string(index=False))

tau: 0.93
test FPR: 0.0095
    family  TPR@1%FPR  AUROC
OpenAI GPT      0.196  0.815
     LLaMA      0.207  0.810
  GLM-130B      0.260  0.839
   FLAN-T5      0.065  0.676
       OPT      0.370  0.885
BigScience      0.225  0.775
EleutherAI      0.406  0.901


C is chosen by accuracy on the tuning slice, as in [53]. Accuracy rises steeply up to C = 8 (0.889) and then flattens: each doubling from 16 to 64 adds about a tenth of a point, less than the standard error of accuracy on 11,030 tuning texts (about 0.3 points). C = 64, the top of the original grid, is therefore kept rather than extending the grid.

τ* is 0.93, far above the default cut-off of 0.5, because 1% of validation human texts already receive a machine probability above 0.93. On the unseen test humans it flags 0.95%, so τ* holds. Tested on each family, the in-distribution model reaches AUROC from 0.676 (FLAN-T5) to 0.901 (EleutherAI), yet catches only 6.5% to 40.6% of machine text at 1% FPR. The gap between a respectable AUROC and a low TPR at the operating point appears before any family is withheld, and is the reference against which LOFO is compared.

## 4. All training sets
Every training set is fitted once per seed: 8 sets × 5 seeds = 40 runs. Each run saves its scores on the tuning slice, the calibration humans and the whole test split, so results can be computed without refitting, and the fusion can reuse them.

In [6]:
# 1. Where each run's scores go, and the texts every model is scored on
SCORES_DIR = MAGE_DIR / "scores" / "lexical"
SCORES_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
SET_NAMES = ["In-distribution"] + [f"LOFO {family}" for family in families]
test = test_set(corpus)

# 2. Fit, score and save each run, skipping any already saved
for seed in SEEDS:
    for set_name in SET_NAMES:
        out_path = SCORES_DIR / f"seed{seed}_{set_name.replace(' ', '_')}.parquet"
        if out_path.exists():
            continue

        train_df = load_training_set(corpus, membership, seed=seed, name=set_name)
        train_part = train_df[train_df["part"] == "train"]
        tune_part = train_df[train_df["part"] == "tune"]
        model, best_c, accuracies = fit_lexical_baseline(
            train_part["text"], train_part["label"], tune_part["text"], tune_part["label"], seed=seed)

        # Score = probability of machine, on every text the later steps need
        scores = pd.concat([
            pd.DataFrame({"text_id": tune_part["text_id"], "part": "tune",
                          "score": model.predict_proba(tune_part["text"])[:, 1]}),
            pd.DataFrame({"text_id": calibration["text_id"], "part": "calibration",
                          "score": model.predict_proba(calibration["text"])[:, 1]}),
            pd.DataFrame({"text_id": test["text_id"], "part": "test",
                          "score": model.predict_proba(test["text"])[:, 1]}),
        ])
        scores.assign(seed=seed, training_set=set_name, c=best_c).to_parquet(out_path, index=False)
        print(f"seed {seed}, {set_name}: C = {best_c:g}, tune accuracy {accuracies[best_c]:.4f}")

seed 0, In-distribution: C = 64, tune accuracy 0.8939
seed 0, LOFO OpenAI GPT: C = 32, tune accuracy 0.9073
seed 0, LOFO LLaMA: C = 32, tune accuracy 0.9086
seed 0, LOFO GLM-130B: C = 16, tune accuracy 0.9049
seed 0, LOFO FLAN-T5: C = 64, tune accuracy 0.9051
seed 0, LOFO OPT: C = 16, tune accuracy 0.8814
seed 0, LOFO BigScience: C = 32, tune accuracy 0.9040
seed 0, LOFO EleutherAI: C = 32, tune accuracy 0.9020
seed 1, In-distribution: C = 32, tune accuracy 0.9011
seed 1, LOFO OpenAI GPT: C = 16, tune accuracy 0.9031
seed 1, LOFO LLaMA: C = 32, tune accuracy 0.9161
seed 1, LOFO GLM-130B: C = 32, tune accuracy 0.9094
seed 1, LOFO FLAN-T5: C = 32, tune accuracy 0.9054
seed 1, LOFO OPT: C = 16, tune accuracy 0.8914
seed 1, LOFO BigScience: C = 64, tune accuracy 0.9008
seed 1, LOFO EleutherAI: C = 32, tune accuracy 0.9028
seed 2, In-distribution: C = 32, tune accuracy 0.8937
seed 2, LOFO OpenAI GPT: C = 64, tune accuracy 0.9039
seed 2, LOFO LLaMA: C = 32, tune accuracy 0.9109
seed 2, LOFO 

## 5. Results
For every run, τ* is set on its calibration scores and applied to the test split. The in-distribution model is tested on every family; each LOFO model only on the family it never saw.

In [5]:
SCORES_DIR = MAGE_DIR / "scores" / "lexical"
SCORES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Every saved run, with each text's label and family
runs = pd.concat([pd.read_parquet(path) for path in sorted(SCORES_DIR.glob("*.parquet"))])
runs = runs.merge(corpus[["text_id", "label", "family"]], on="text_id")

# 2. TPR and AUROC per run and target family
rows = []
for (seed, set_name), run in runs.groupby(["seed", "training_set"]):
    calibration_part = run[run["part"] == "calibration"]
    test_part = run[run["part"] == "test"]
    tau = calibrate_threshold(calibration_part["score"], calibration_part["label"])
    test_fpr = fpr_at_threshold(test_part["score"], test_part["label"], tau)

    # In-distribution is tested on every family; a LOFO model only on its held-out family
    if set_name == "In-distribution":
        condition, targets = "In-distribution", families
    else:
        condition, targets = "LOFO", [set_name.removeprefix("LOFO ")]

    for family in targets:
        family_test = test_part[(test_part["label"] == 0) | (test_part["family"] == family)]
        rows.append({"seed": seed, "condition": condition, "family": family,
                     "tpr": tpr_at_threshold(family_test["score"], family_test["label"], tau),
                     "auroc": auroc(family_test["score"], family_test["label"]),
                     "test_fpr": test_fpr})
results = pd.DataFrame(rows)

# 3. Per family, in-distribution against LOFO: mean over the five seeds, then the spread
print(results.groupby(["family", "condition"])[["tpr", "auroc"]].mean().unstack("condition").round(3).to_string())
print(results.groupby(["family", "condition"])[["tpr", "auroc"]].std().unstack("condition").round(3).to_string())

# 4. Mean and worst-case TPR across the seven families, per seed, then averaged over seeds
per_seed = results.groupby(["condition", "seed"])["tpr"].agg(mean_tpr="mean", worst_tpr="min")
print(per_seed.groupby("condition").agg(["mean", "std"]).round(3).to_string())

# 5. Does τ* hold on the test humans in every run?
print(results.groupby("condition")["test_fpr"].agg(["min", "max"]).round(4).to_string())

                       tpr                  auroc       
condition  In-distribution   LOFO In-distribution   LOFO
family                                                  
BigScience           0.238  0.236           0.781  0.768
EleutherAI           0.435  0.398           0.908  0.897
FLAN-T5              0.072  0.070           0.681  0.671
GLM-130B             0.247  0.255           0.838  0.840
LLaMA                0.207  0.103           0.805  0.719
OPT                  0.386  0.216           0.889  0.833
OpenAI GPT           0.207  0.102           0.815  0.735
                       tpr                  auroc       
condition  In-distribution   LOFO In-distribution   LOFO
family                                                  
BigScience           0.009  0.005           0.005  0.001
EleutherAI           0.019  0.007           0.004  0.001
FLAN-T5              0.005  0.002           0.005  0.002
GLM-130B             0.010  0.008           0.002  0.003
LLaMA                0.005  0.0

τ* holds across all 40 runs: test FPR stays between 0.95% and 1.19%. Averaged over seeds, holding a family out lowers mean TPR across the seven families from 0.256 to 0.196, far beyond two seed standard deviations (0.006 and 0.004). The loss is concentrated in three families, each beyond twice the larger seed standard deviation: OPT (0.386 to 0.216), OpenAI GPT (0.207 to 0.102) and LLaMA (0.207 to 0.103). EleutherAI drops from 0.435 to 0.393, just beyond that threshold, while BigScience, FLAN-T5 and GLM-130B change by less than it.

For the three families that drop, TPR falls by 44% to 51% of its in-distribution value, but AUROC by only 6% to 11% (OPT 0.889 to 0.833, OpenAI GPT 0.815 to 0.735, LLaMA 0.805 to 0.719), so AUROC understates the loss at the operating point, as H1 predicts. AUROC also overstates reliability without any loss: under LOFO, GLM-130B reaches an AUROC of 0.840 while only 25.5% of its texts are caught.

Worst-case TPR barely changes (0.072 to 0.070) because the worst family is FLAN-T5 in both conditions, which the lexical baseline barely detects even when trained on it. For this detector, the worst case reflects a family it cannot detect, not a loss under shift.

## 6. Save
The per-run results are saved for the analysis chapter; the scores saved in section 4 are kept for the fusion.

In [10]:
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# One row per seed, condition and target family: 35 in-distribution and 35 LOFO
results.to_csv(RESULTS_DIR / "lexical.csv", index=False)
print("rows:", len(results), "| saved to", RESULTS_DIR / "lexical.csv")

rows: 70 | saved to /Users/xwm517/Desktop/logos/data/results/lexical.csv


In [11]:
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

summary = (
    results.assign(detector="lexical")
    .groupby(["detector", "family", "condition"])[["tpr", "auroc", "test_fpr"]]
    .agg(["mean", "std"])
)
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.round(4).reset_index()

summary.to_csv(RESULTS_DIR / "summary_lexical.csv", index=False)
print(summary)

   detector      family        condition  tpr_mean  tpr_std  auroc_mean  \
0   lexical  BigScience  In-distribution    0.2380   0.0087      0.7808   
1   lexical  BigScience             LOFO    0.2360   0.0047      0.7677   
2   lexical  EleutherAI  In-distribution    0.4352   0.0192      0.9078   
3   lexical  EleutherAI             LOFO    0.3985   0.0071      0.8973   
4   lexical     FLAN-T5  In-distribution    0.0720   0.0053      0.6813   
5   lexical     FLAN-T5             LOFO    0.0704   0.0024      0.6714   
6   lexical    GLM-130B  In-distribution    0.2472   0.0095      0.8380   
7   lexical    GLM-130B             LOFO    0.2555   0.0079      0.8402   
8   lexical       LLaMA  In-distribution    0.2073   0.0054      0.8053   
9   lexical       LLaMA             LOFO    0.1025   0.0055      0.7188   
10  lexical         OPT  In-distribution    0.3862   0.0103      0.8893   
11  lexical         OPT             LOFO    0.2157   0.0039      0.8334   
12  lexical  OpenAI GPT  